In [2]:
import pandas as pd
import openpyxl

In [3]:
df = pd.read_excel("/home/callum/sno_data/raw/wang.xlsx", header = 0)

df = df.rename(columns = {"Protein accession": "id", "S-nitrosylation position": "position", "Amino acid" : "aa"})
print(df.head())

# this study applied fdr filter of 0.01 to peptides

              id  position aa  \
0  TGME49_200250       309  C   
1  TGME49_200320       125  C   
2  TGME49_201130       673  C   
3  TGME49_201400       546  C   
4  TGME49_201630        77  C   

                                 Protein description  
0                          microneme protein MIC17A   
1  hypoxanthine-xanthine-guanine phosphoribosyl t...  
2               rhoptry kinase family protein ROP33   
3                 Sin3-associated polypeptide SAP18   
4  protein phosphatase 2C domain-containing protein   


In [4]:
import io
import pandas as pd
import requests
# needed script to convert these IDS to uniprot
# --- your input: edit this ---
df = df

TAXID = 508771  # T. gondii ME49
URL = "https://rest.uniprot.org/uniprotkb/stream"

def fetch(ids):
    q = "(" + " OR ".join(f"gene:{i}" for i in ids) + f") AND (organism_id:{TAXID})"
    r = requests.get(URL, params={
        "query": q, "format": "tsv",
        "fields": "accession,reviewed,gene_names,length,sequence"}, timeout=60)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), sep="\t")

ids = df["id"].drop_duplicates().tolist()
hits = pd.concat([fetch(ids[i:i+50]) for i in range(0, len(ids), 50)], ignore_index=True)

# pull the TGME49_ ID out of the gene names column
hits["id"] = hits["Gene Names"].fillna("").str.findall(r"TGME49_\d+")
hits = hits.explode("id").dropna(subset=["id"])
hits = hits[hits["id"].isin(ids)]

# one entry per gene: reviewed first, then longest
hits["is_reviewed"] = hits["Reviewed"].eq("reviewed")
hits = hits.sort_values(["id", "is_reviewed", "Length"], ascending=[True, False, False])
n_cand = hits["id"].value_counts()
best = hits.drop_duplicates("id").rename(columns={"Entry": "uniprot_id", "Sequence": "seq"})
best["n_candidates"] = best["id"].map(n_cand)

out = df.merge(best[["id", "uniprot_id", "Reviewed", "Length", "n_candidates", "seq"]],
               on="id", how="left")

# residue at your position, plus a +/-3 window for context
def residue(row):
    s, p = row["seq"], int(row["position"])
    if not isinstance(s, str) or not (0 < p <= len(s)):
        return pd.Series([None, None])
    window = s[max(0, p-4):p-1] + "[" + s[p-1] + "]" + s[p:p+3]
    return pd.Series([s[p-1], window])

out[["residue", "window"]] = out.apply(residue, axis=1)

# if you have an aa column, check it (and whether pos +/-1 would match)
if "aa" in out.columns:
    out["residue_ok"] = out["residue"] == out["aa"]
    def offset(row):
        s, p = row["seq"], int(row["position"])
        if row["residue_ok"] or not isinstance(s, str):
            return None
        return next((d for d in (-1, 1) if 0 < p+d <= len(s) and s[p+d-1] == row["aa"]), None)
    out["offset_match"] = out.apply(offset, axis=1)

print(f"Mapped {out['uniprot_id'].notna().sum()} / {len(out)} rows")
print("Unmapped:", out.loc[out["uniprot_id"].isna(), "id"].unique().tolist())
out.drop(columns="seq")

Mapped 978 / 983 rows
Unmapped: ['TGME49_219850', 'TGME49_224350', 'TGME49_255260', 'TGME49_297060']


,id,position,aa,Protein description,uniprot_id,Reviewed,Length,n_candidates,residue,window,residue_ok,offset_match
0,TGME49_200250,309,C,microneme protein MIC17A,S8GIC9,unreviewed,351.0,1.0,C,FES[C]QMK,True,None
1,TGME49_200320,125,C,hypoxanthine-xanthine-guanine phosphoribosyl t...,A0A125YSF4,unreviewed,279.0,1.0,C,HII[C]ILK,True,None
2,TGME49_201130,673,C,rhoptry kinase family protein ROP33,S8GP43,unreviewed,746.0,1.0,C,LGV[C]LKQ,True,None
3,TGME49_201400,546,C,Sin3-associated polypeptide SAP18,S8F3I3,unreviewed,868.0,1.0,C,KDV[C]WEA,True,None
4,TGME49_201630,77,C,protein phosphatase 2C domain-containing protein,S8GP24,unreviewed,372.0,1.0,C,EDA[C]VVT,True,None
...,...,...,...,...,...,...,...,...,...,...,...,...
978,TGME49_321440,672,C,SWI2/SNF2 ISWI-like SANT,S8GK42,unreviewed,1249.0,1.0,C,EDY[C]RIR,True,None
979,TGME49_321500,130,C,RNA recognition motif-containing protein,S8FBK6,unreviewed,475.0,1.0,C,TSS[C]EPG,True,None
980,TGME49_321530,260,C,cathepsin CPL,A0A125YST9,unreviewed,422.0,1.0,C,LMD[C]SRA,True,None
981,TGME49_321620,471,C,dynamin-related protein DRPB,A0A125YT62,unreviewed,860.0,1.0,C,VIE[C]LDR,True,None


In [5]:
# filter out mismatches
subset = out[out["residue_ok"] == True]
print(subset.shape)
print(out.shape)

(978, 13)
(983, 13)


In [9]:
subset = out[["uniprot_id", "position"]]

subset = subset.rename(columns = {"uniprot_id": "id"})

# add organism column
subset["organism"] = "Toxoplasma gondii"
print(subset.head())

           id  position           organism
0      S8GIC9       309  Toxoplasma gondii
1  A0A125YSF4       125  Toxoplasma gondii
2      S8GP43       673  Toxoplasma gondii
3      S8F3I3       546  Toxoplasma gondii
4      S8GP24        77  Toxoplasma gondii


In [10]:
# save the processed df
subset.to_csv("/home/callum/sno_data/processed/wang/wang_org.csv", index = False)
